# README

In this notebook:
- We check whether the directories contain the same number of packages and same content.
- Define the buckets that will serve as groups in our experiments. We are not comparing packages with different sizes, thus size it is a blocking factor.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from pathlib import PosixPath

In [2]:
#algos = [
#    'gzip', 'xz', 'zstd', 'brotli', 'zopfli', 'tar'
#]
# (Folders, Extension)
EXTENSIONS = {
    'npm' : '.tgz',
    'xz'  : '.tar.xz'
}

In [3]:
def return_dir_files(dirname : str):
    '''
    Returns all the files from a specified @dirpath 
    and collects all the files with the expected extension. 
    '''
    if dirname not in EXTENSIONS:
        raise KeyError(f"no extension configured for {dirname!r}")
    target_extension = EXTENSIONS[dirname]
    
    dirpath = Path(dirname)
    if not dirpath.is_dir():
        raise FileNotFoundError(f"directory {dirpath} not found")
    # collects files with expected extension
    return [
        filepath for filepath in dirpath.iterdir()
        if filepath.is_file() and filepath.name.endswith(target_extension)
    ] 

In [4]:
def true_stem(p: Path) -> str:
    name = p.name
    for ext in list(EXTENSIONS.values()):
        if name.endswith(ext):
            return name[: -len(ext)]
    return p.stem

In [5]:
def get_intersection(array : list[list[Path]]):
    '''returns elements in common in list of list of paths'''
    sets = [set(map(lambda p: true_stem(p), l)) for l in array]
    return list(set.intersection(*sets))

In [6]:
def get_sizes(files : list[Path]): 
    return [
        {'name': f, 'size' : f.stat().st_size} for f in files
    ]

In [7]:
allfiles = {
    key : return_dir_files(key) for key in EXTENSIONS
}

sizes = {
    key : pd.DataFrame(get_sizes(allfiles[key])) for key in allfiles
}

# list of strings
intersection = get_intersection(list(allfiles.values()))

FileNotFoundError: directory npm not found

In [ ]:
# same packages across populations
for k in sizes:
    sizes[k]['_key'] = sizes[k]['name'].apply(true_stem)
    sizes[k] = sizes[k][sizes[k]['_key'].isin(intersection)]

# Buckets

In [ ]:
def analyze_size_buckets(df, n_buckets=120):
    df = df.copy()
    df['log_size'] = np.log10(df['size'])
    df['bucket'] = pd.qcut(df['log_size'], q=n_buckets, labels=False, duplicates='drop')
    grp = df.groupby('bucket', observed=True)['size'].agg(['count', 'min', 'max', 'mean', 'std'])
    grp['cv'] = grp['std'] / grp['mean']
    return df, grp

In [ ]:
# make the buckets according to the original compression
bucket_df, buckets = analyze_size_buckets(sizes['xz'])

In [ ]:
max_value = int(max(bucket_df['bucket']))
sample = bucket_df[bucket_df['bucket'] == max_value]
sample

In [ ]:
sample.to_csv("sample.csv", index=False)